In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import yaml
from config import DAS_FILE
from DAS import DAS

with open("configs/denoising.yaml") as f:
    cfg = yaml.safe_load(f)

# Load sensor metadata; yaml values take priority over file metadata
_meta = DAS(DAS_FILE).meta
dx = cfg.get("dx") or _meta["dx"]
dt = cfg.get("dt") or _meta["dt"]

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from models.unet import UNet
from preprocessing import make_preprocess
from dataset import DASSampleDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_DIR = cfg.get("data_dir", "data")
SAVE_DIR = cfg.get("save_dir", "results/denoising")
FS_DAS = cfg["fs"]

df = pd.read_csv(os.path.join(DATA_DIR, cfg["labels_csv"]))
df_pos = df[df["count"] > 0].reset_index(drop=True)

with open(os.path.join(SAVE_DIR, "splits.json")) as f:
    splits = json.load(f)

test_df = df_pos[df_pos["sample_id"].isin(splits["test"])].reset_index(drop=True)
print(f"Test set: {len(test_df)} samples")
display(test_df[["sample_id", "count", "vehicle_type"]])

In [5]:
test_df

,sample_id,data_path,count,start_frame,end_frame,vehicle_type
57,32,data/sample_000032.npy,1,2832,2983,van
58,108,data/sample_000108.npy,1,5111,5262,truck
59,79,data/sample_000079.npy,4,4242,4392,mixed
60,47,data/sample_000047.npy,4,3282,3433,mixed
61,29,data/sample_000029.npy,3,2742,2893,mixed
62,53,data/sample_000053.npy,2,3462,3613,mixed
63,83,data/sample_000083.npy,1,4361,4512,van
64,1,data/sample_000001.npy,4,1902,2053,mixed
65,67,data/sample_000067.npy,1,3882,4033,suv
66,21,data/sample_000021.npy,1,2502,2653,suv


In [ ]:
steps = [(s["name"], {k: v for k, v in s.items() if k != "name"}) for s in cfg["steps"]]
pp = make_preprocess(steps=steps, dx=dx, dt=dt)

test_dataset = DASSampleDataset(test_df, preprocess=pp, fs_das=FS_DAS)
test_loader  = DataLoader(test_dataset, batch_size=2, shuffle=False)
print(f"Test samples: {len(test_dataset)}")

In [ ]:
model = UNet(in_channels=1, out_channels=1).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

In [ ]:
# Load the trained model saved by: python train.py --task denoising
model.load_state_dict(torch.load("results/denoising/best_model.pt", map_location=device))
model.eval()
print("Model loaded from results/denoising/best_model.pt")

In [ ]:
from Utilities import plot_das_data
import matplotlib.pyplot as plt

SAMPLE_IDX = 0  # change to inspect different test samples

row = test_df.iloc[SAMPLE_IDX]
sid = row["sample_id"]
raw = np.load(row["data_path"]).astype(np.float32)
clean = pp(raw, FS_DAS).astype(np.float32)

with torch.no_grad():
    raw_t = torch.from_numpy(raw[None, None, ...]).to(device)
    pred = model(raw_t).squeeze().cpu().numpy()

channels = np.arange(raw.shape[0])
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
plot_das_data(data=raw,   channels=channels, dx=dx, dt=dt, title=f"Sample {sid} — Raw",           ax=axes[0], fig=fig, show=False)
plot_das_data(data=clean, channels=channels, dx=dx, dt=dt, title=f"Sample {sid} — Preprocessed",  ax=axes[1], fig=fig, show=False)
plot_das_data(data=pred,  channels=channels, dx=dx, dt=dt, title=f"Sample {sid} — UNet Denoised", ax=axes[2], fig=fig, show=False)
plt.tight_layout()
plt.show()

In [ ]:
from Utilities import plot_single
import matplotlib.pyplot as plt

CHANNEL = 5  # channel to inspect
T0, T1 = 0, 5

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

plot_single(data=raw,   channel_num=CHANNEL, dx=dx, dt=dt, start_time=T0, end_time=T1, ax=axes[0], show=False)
axes[0].set_title(f"Sample {sid} — Raw | Channel {CHANNEL}")

plot_single(data=clean, channel_num=CHANNEL, dx=dx, dt=dt, start_time=T0, end_time=T1, ax=axes[1], show=False)
axes[1].set_title(f"Sample {sid} — Preprocessed | Channel {CHANNEL}")

plot_single(data=pred,  channel_num=CHANNEL, dx=dx, dt=dt, start_time=T0, end_time=T1, ax=axes[2], show=False)
axes[2].set_title(f"Sample {sid} — UNet Denoised | Channel {CHANNEL}")

plt.tight_layout()
plt.show()

In [ ]:
# Run `python predict.py --task denoising` to regenerate data/denoised/
print("Denoised samples → data/denoised/  |  Run: python predict.py --task denoising")